In [1]:
!pip install dagshub mlflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 1.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 73.9 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 69.4 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 52.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [3]:
import pandas as pd
import numpy as np
import dagshub
import mlflow
import mlflow.sklearn 
import time

dagshub.init(repo_owner='tvada22', repo_name='ML--Assignment2-IEEE-CIS-Fraud-Detection', mlflow=True)

def reduce_mem_usage(df):
    start_mem = df.memory_usage().sum() / 1024**2
    print(f'memory usage of dataframe is {start_mem:.2f} MB')
    for col in df.columns:
        col_type = df[col].dtype
        if col_type != object:
            c_min = df[col].min()
            c_max = df[col].max()
            if str(col_type)[:3] == 'int':
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
                elif c_min > np.iinfo(np.int64).min and c_max < np.iinfo(np.int64).max:
                    df[col] = df[col].astype(np.int64)  
            else:
                if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
                    df[col] = df[col].astype(np.float16)
                elif c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                    df[col] = df[col].astype(np.float32)
                else:
                    df[col] = df[col].astype(np.float64)
    end_mem = df.memory_usage().sum() / 1024**2
    print(f'memory usage after optimization is: {end_mem:.2f} MB')
    return df

test_df = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv')
test_df = reduce_mem_usage(test_df)

submission_ids = test_df['TransactionID']
X_test_raw = test_df.drop(columns=['TransactionID', 'TransactionDT'])

model_name = "GradientBoosting_Final_Model"
exact_uri = "mlflow-artifacts:/fbd409e71b074257ad19db2bfbc51767/c2ca64422fae40f0866c71a038fffd34/artifacts/MLmodel"

try:
    mlflow.register_model(model_uri=exact_uri, name=model_name)
except Exception as e:
    print(f"Model registration note: {e}")

time.sleep(5) 

registry_uri = f"models:/{model_name}/latest"

try:
    loaded_pipeline = mlflow.sklearn.load_model(registry_uri)
    
except Exception as e:
    print(f"\n[!] DagsHub Server Error encountered when resolving registry link.")
    print("[!] Fallback triggered: Loading model directly from source artifact to complete inference...")
    loaded_pipeline = mlflow.sklearn.load_model(exact_uri)

submission = pd.DataFrame({
    'TransactionID': submission_ids,
    'isFraud': pred_probs
})

submission.to_csv('/kaggle/working/submission.csv', index=False)

Initialized MLflow to track repo "tvada22/ML--Assignment2-IEEE-CIS-Fraud-Detection"

Repository tvada22/ML--Assignment2-IEEE-CIS-Fraud-Detection initialized!

memory usage of dataframe is 1519.24 MB


/tmp/ipykernel_57/154742219.py:28: RuntimeWarning: overflow encountered in cast
  if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
/tmp/ipykernel_57/154742219.py:28: RuntimeWarning: overflow encountered in cast
  if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
/tmp/ipykernel_57/154742219.py:28: RuntimeWarning: overflow encountered in cast
  if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
/tmp/ipykernel_57/154742219.py:28: RuntimeWarning: overflow encountered in cast
  if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
/tmp/ipykernel_57/154742219.py:28: RuntimeWarning: overflow encountered in cast
  if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
/tmp/ipykernel_57/154742219.py:28: RuntimeWarning: overflow encountered in cast
  if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
/tmp/ipykernel_57/154742219.py:28: RuntimeWarning: overflow enco

memory usage after optimization is: 472.59 MB


Successfully registered model 'XGBoost_Final_Model'.
2026/05/04 22:08:22 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: XGBoost_Final_Model, version 1
Created version '1' of model 'XGBoost_Final_Model'.



[!] DagsHub Server Error encountered when resolving registry link.
[!] Fallback triggered: Loading model directly from source artifact to complete inference...


MlflowException: The following failures occurred while downloading one or more artifacts from https://dagshub.com/tvada22/ML--Assignment2-IEEE-CIS-Fraud-Detection.mlflow/api/2.0/mlflow-artifacts/artifacts/ea4d3cc287c54165875c2175234e0545/c82333fd091342e6b02cb8c94771c95f/artifacts:
##### File MLmodel #####
API request to https://dagshub.com/tvada22/ML--Assignment2-IEEE-CIS-Fraud-Detection.mlflow/api/2.0/mlflow-artifacts/artifacts/ea4d3cc287c54165875c2175234e0545/c82333fd091342e6b02cb8c94771c95f/artifacts/MLmodel failed with exception HTTPSConnectionPool(host='dagshub.com', port=443): Max retries exceeded with url: /tvada22/ML--Assignment2-IEEE-CIS-Fraud-Detection.mlflow/api/2.0/mlflow-artifacts/artifacts/ea4d3cc287c54165875c2175234e0545/c82333fd091342e6b02cb8c94771c95f/artifacts/MLmodel (Caused by ResponseError('too many 500 error responses'))